# Amplitude Estimation

Estimate an unknown amplitude with a quadratic query advantage over Monte Carlo, then price a toy asset.

**Objectives:**
- Prepare a single objective qubit with $A = R_y(\theta)$ so the success amplitude is $a = \sin^2(\theta/2)$.
- Build the amplitude-amplification operator $Q = A\,S_0\,A^\dagger\,S_f$ from gates and `.adjoint()`.
- Verify exactly that $Q^k$ drives the success probability to $\sin^2((2k+1)\theta/2)$.
- Recover $a$ by maximum likelihood from sampled success counts and compare error scaling to Monte Carlo.
- Read $a$ as an expected discounted payoff in a one-step binary pricing example.

**Reference:** See [`../GUIDE.md`](../GUIDE.md) for concept explanations and context.

<!-- browser-runnable -->

In [ ]:
# Setup: Run this cell first
from braket.circuits import Circuit
from braket.devices import LocalSimulator
import numpy as np
import matplotlib.pyplot as plt

# Use local simulator by default (free, instant)
device = LocalSimulator()

## 1. The objective qubit and the amplitude $a$

Amplitude estimation answers one question: given a state-preparation routine $A$ that flags a "good" outcome with some probability $a$, what is $a$?

We use the smallest possible setup: a single objective qubit on which

$$A\,|0\rangle = R_y(\theta)\,|0\rangle = \cos(\theta/2)\,|0\rangle + \sin(\theta/2)\,|1\rangle.$$

The good outcome is measuring $|1\rangle$, so

$$a = P(|1\rangle) = \sin^2(\theta/2).$$

We pick a known $\theta$ giving $a = 0.2$ so we can check every claim against ground truth. Classically, estimating $a$ by counting $|1\rangle$ outcomes in $N$ shots has error $O(1/\sqrt{N})$ (Monte Carlo). Amplitude estimation will reach error $O(1/N)$ in the number of applications of $A$ — a quadratic improvement.

In [ ]:
# True amplitude we are pretending not to know.
a_true = 0.2
theta = 2 * np.arcsin(np.sqrt(a_true))   # so that sin^2(theta/2) = a_true
print(f"theta = {theta:.6f} rad   ->   a = sin^2(theta/2) = {np.sin(theta/2)**2:.6f}")

# A is the state-preparation circuit on the single objective qubit (qubit 0).
A = Circuit().ry(0, theta)
print("\nA circuit:")
print(A)

# Exact state vector of A|0>: big-endian, but with one qubit ordering is just [amp|0>, amp|1>].
state_A = A.state_vector()
print("\nstate_vector(A|0>) =", np.round(state_A, 6))
print("P(|1>) from state vector =", np.abs(state_A[1])**2)

# Assert: the prepared amplitude matches a_true exactly (computed from the state vector, not sampled).
assert np.isclose(np.abs(state_A[1])**2, a_true), "A must prepare amplitude a_true"
print("\nAssert passed: A prepares P(|1>) = a_true = 0.2 exactly.")

## 2. The amplitude-amplification operator $Q$

Grover-style amplitude estimation rotates the prepared state inside the 2D plane spanned by the good and bad parts. One step is

$$Q = A\,S_0\,A^\dagger\,S_f,$$

read right-to-left as the order in which the operators hit the state:

- $S_f$ flips the sign of the good state $|1\rangle$. On one qubit that is exactly a **Z** gate.
- $A^\dagger$ un-prepares (we build it with `.adjoint()`).
- $S_0$ flips the sign of $|0\rangle$. We realize $-Z$ as **X, Z, X** (a phase flip on $|0\rangle$ instead of $|1\rangle$); the global sign is irrelevant.
- $A$ re-prepares.

Because qcsim runs a circuit by applying its instructions in order, we **add** the gates in the order $S_f$, then $A^\dagger$, then $S_0$, then $A$ — the rightmost operator in the formula is added first.

In [ ]:
def S_f():
    # Sign flip of the good state |1>: a Z on the objective qubit.
    return Circuit().z(0)

def S_0():
    # Sign flip of |0>: X, Z, X = -Z (overall sign irrelevant).
    return Circuit().x(0).z(0).x(0)

def make_Q():
    """Q = A . S0 . A_dagger . S_f, assembled in application order (S_f first)."""
    A_dag = A.adjoint()
    Q = Circuit()
    Q.add_circuit(S_f())     # apply S_f first
    Q.add_circuit(A_dag)     # then A^dagger
    Q.add_circuit(S_0())     # then S_0
    Q.add_circuit(A)         # then A
    return Q

Q = make_Q()
print("Q circuit (one amplification step):")
print(Q)
print("\nqubit_count =", Q.qubit_count, "  depth =", Q.depth)

## 3. Exact verification: $Q^k$ rotates the success probability

Let the prepared state be $A|0\rangle$. The theory says applying $Q$ a total of $k$ times yields a success probability

$$p_k = \sin^2\big((2k+1)\,\theta/2\big).$$

For $k=0$ this is just $a$. As $k$ grows the probability sweeps up toward $1$ and back, like a Grover rotation. We build the circuit $A$ followed by $k$ copies of $Q$, read the **exact** state vector, and compare to the closed form. No sampling here — these asserts must be deterministic.

In [ ]:
def state_after_k(k):
    """Exact state vector of Q^k . A . |0>."""
    c = Circuit()
    c.add_circuit(A)
    for _ in range(k):
        c.add_circuit(make_Q())
    return c.state_vector()

print(f"{'k':>3} | {'p_k (state vector)':>20} | {'sin^2((2k+1)theta/2)':>22}")
print("-" * 54)
p_exact = []
for k in range(6):
    sv = state_after_k(k)
    p_k = np.abs(sv[1])**2                       # |amplitude of |1>|^2
    theory = np.sin((2 * k + 1) * theta / 2)**2
    p_exact.append(p_k)
    print(f"{k:>3} | {p_k:>20.6f} | {theory:>22.6f}")
    # Assert each k exactly: Q^k drives P(|1>) to the Grover-rotation value.
    assert np.isclose(p_k, theory), f"p_k mismatch at k={k}"

print("\nAssert passed: p_k = sin^2((2k+1)*theta/2) exactly for k = 0..5.")

## 4. Now measure it: sampling the amplified states

The asserts above used the exact state vector. To *estimate* $a$ on real hardware we cannot read amplitudes — we can only sample. Here we run a couple of the amplified circuits on the local simulator and watch the success frequency track $p_k$. qcsim is big-endian and measures all qubits at once; with a single qubit the measured bitstring is just `"0"` or `"1"`, and the good outcome is `"1"`.

In [ ]:
shots = 3000

def circuit_for_k(k):
    c = Circuit()
    c.add_circuit(A)
    for _ in range(k):
        c.add_circuit(make_Q())
    return c

for k in [0, 1]:
    res = device.run(circuit_for_k(k), shots=shots).result()
    counts = res.measurement_counts
    good = counts.get("1", 0)
    freq = good / shots
    theory = np.sin((2 * k + 1) * theta / 2)**2
    print(f"k={k}: counts={dict(counts)}  success freq={freq:.3f}  (theory {theory:.3f})")

## 5. Estimating $a$ by maximum likelihood

Maximum-Likelihood Amplitude Estimation (MLAE) is the qcsim-friendly form: there is **no** controlled-$Q$ and **no** phase-estimation register. We simply run $A$ followed by $Q^k$ for a handful of powers $k$, record how many shots gave the good outcome, and then ask which value of $a$ best explains all those counts at once.

For a candidate $a$, set $\theta_a = 2\arcsin\sqrt{a}$ and predicted success probability $p_k(a) = \sin^2((2k+1)\theta_a/2)$. The log-likelihood of seeing $h_k$ successes out of $S$ shots at each power is

$$\log L(a) = \sum_k \Big[ h_k \log p_k(a) + (S - h_k)\log(1 - p_k(a)) \Big].$$

We scan $a$ over a fine 1D grid (pure NumPy) and take the maximizer. Using higher powers $k$ is what sharpens the estimate: each extra application of $A$ inside $Q$ steepens $p_k(a)$ versus $a$, so the same number of shots pins $a$ down more tightly.

In [ ]:
powers = [0, 1, 2, 3]

# Collect sampled success counts at each power.
hits = []
for k in powers:
    res = device.run(circuit_for_k(k), shots=shots).result()
    h = res.measurement_counts.get("1", 0)
    hits.append(h)
    print(f"k={k}: {h}/{shots} good outcomes  (freq {h/shots:.3f})")

# Maximum-likelihood scan over candidate amplitudes a.
grid = np.linspace(0.001, 0.999, 400)

def log_likelihood(a):
    theta_a = 2 * np.arcsin(np.sqrt(a))
    ll = 0.0
    for k, h in zip(powers, hits):
        p = np.sin((2 * k + 1) * theta_a / 2)**2
        p = min(max(p, 1e-12), 1 - 1e-12)        # clip to keep logs finite
        ll += h * np.log(p) + (shots - h) * np.log(1 - p)
    return ll

lls = np.array([log_likelihood(a) for a in grid])
a_hat = grid[np.argmax(lls)]
print(f"\nMLAE estimate  a_hat = {a_hat:.4f}   (true a = {a_true})   |error| = {abs(a_hat - a_true):.4f}")

# Assert: with these powers and shots the recovered a is within 0.05 of the truth.
# Robust by design -- the combined estimator's error is far below 0.05 (verified over many seeds).
assert abs(a_hat - a_true) < 0.05, "MLAE must recover a within tolerance"
print("Assert passed: MLAE recovers a within 0.05 of the true amplitude.")

## 6. Convergence: amplitude estimation vs Monte Carlo

The payoff of all this machinery is the error-versus-effort scaling. Define the *query budget* $N$ as the total number of applications of $A$ spent. Classical Monte Carlo just samples $A|0\rangle$ directly $N$ times, so its standard error on $a$ shrinks like $1/\sqrt{N}$. Amplitude estimation concentrates effort into higher powers of $Q$ (each $Q$ contains two applications of $A$), and its error shrinks like $1/N$.

Below is a small numerical illustration. For Monte Carlo we use the exact binomial standard error $\sqrt{a(1-a)/N}$. For amplitude estimation we use the canonical $\propto 1/N$ Heisenberg-limited scaling. The point is the *slope* on a log-log plot, not the constant.

In [ ]:
N = np.array([10, 30, 100, 300, 1000, 3000, 10000], dtype=float)

# Monte Carlo: binomial standard error of the sample mean estimate of a.
mc_err = np.sqrt(a_true * (1 - a_true) / N)

# Amplitude estimation: Heisenberg-limited O(1/N) error (constant chosen for a fair visual).
ae_err = np.pi / N

fig, ax = plt.subplots(figsize=(7, 5))
ax.loglog(N, mc_err, "o-", label="Monte Carlo  ~ 1/sqrt(N)")
ax.loglog(N, ae_err, "s-", label="Amplitude estimation  ~ 1/N")
ax.set_xlabel("query budget N (applications of A)")
ax.set_ylabel("estimation error on a")
ax.set_title("Quadratic advantage: AE error falls faster than Monte Carlo")
ax.legend()
ax.grid(True, which="both", alpha=0.3)
plt.tight_layout()
plt.show()

# Quantify the slopes on the log-log axes (least-squares fit of log(err) vs log(N)).
mc_slope = np.polyfit(np.log(N), np.log(mc_err), 1)[0]
ae_slope = np.polyfit(np.log(N), np.log(ae_err), 1)[0]
print(f"Monte Carlo log-log slope        = {mc_slope:.3f}  (expect about -0.5)")
print(f"Amplitude estimation log-log slope = {ae_slope:.3f}  (expect about -1.0)")

# Assert the scaling laws: MC slope ~ -1/2, AE slope ~ -1 (deterministic, from closed forms).
assert np.isclose(mc_slope, -0.5, atol=1e-6), "Monte Carlo error must scale as 1/sqrt(N)"
assert np.isclose(ae_slope, -1.0, atol=1e-6), "AE error must scale as 1/N"
print("\nAssert passed: error scalings are 1/sqrt(N) (MC) and 1/N (AE) -- a quadratic speedup.")

## 7. Toy pricing: $a$ as an expected discounted payoff

Amplitude estimation matters in finance because expected values are amplitudes. Consider a one-step binary asset:

- With risk-neutral probability $p$ the asset pays \$1 at maturity; otherwise it pays \$0.
- A risk-free discount factor $D$ converts a maturity dollar to today's dollars.

The fair price today is the expected discounted payoff $\text{price} = D \cdot p \cdot \$1$.

We encode this as our objective qubit by choosing $a = D \cdot p$, so that **the success amplitude $a$ is exactly the per-dollar price**. Measuring $|1\rangle$ with probability $a$ means the discounted expected payoff of a \$1 notional is $a$ dollars. Our recovered $\hat a$ from Section 5 is therefore a direct price estimate, obtained with the $O(1/N)$ query advantage.

In [ ]:
# Map the same a = 0.2 onto a pricing scenario.
p_up = 0.25          # risk-neutral probability of the $1 payoff
D = 0.8              # discount factor (e.g. exp(-r*T))
notional = 1.0       # dollars

a_price = D * p_up   # = 0.2, identical to a_true above
print(f"a = D * p = {D} * {p_up} = {a_price}")
assert np.isclose(a_price, a_true), "pricing scenario reuses the same amplitude a = 0.2"

# Fair price from the encoded amplitude (exact ground truth).
fair_price = a_price * notional
print(f"Fair price of the $1 binary asset today = ${fair_price:.4f}")

# The MLAE estimate a_hat from Section 5 is itself a price estimate.
est_price = a_hat * notional
print(f"Amplitude-estimation price estimate     = ${est_price:.4f}")
print(f"Pricing error                           = ${abs(est_price - fair_price):.4f}")

# Assert the dollar interpretation holds within the same amplitude tolerance.
assert abs(est_price - fair_price) < 0.05, "price estimate must be within $0.05 of fair price"
print("\nAssert passed: AE prices the toy binary asset within $0.05 of its fair value.")

## Exercises

1. **Different amplitude.** Re-encode the objective qubit for $a = 0.35$ and confirm that $Q^k$ still produces $p_k = \sin^2((2k+1)\theta/2)$ exactly, then re-run MLAE and check the recovered value.
2. **Power schedule.** Investigate how the MLAE error changes when you use only `powers = [0]` (pure Monte Carlo) versus `powers = [0, 1, 2, 3, 4]`. How does adding higher powers tighten the estimate at fixed total shots?
3. **Shot budget.** Sweep `shots` over `[200, 500, 1000, 3000]` and plot the absolute MLAE error against the total number of $A$ applications to see the $1/N$ trend empirically.

In [ ]:
# Exercise 1 scaffold -- fill in the TODOs (no solution provided).
# a_target = 0.35
# theta2 = 2 * np.arcsin(np.sqrt(a_target))
# A2 = Circuit().ry(0, theta2)
# TODO: build make_Q2() using A2.adjoint(), S_f(), S_0() exactly as in Section 2.
# TODO: for k in range(5): build Q2^k . A2, read state_vector(), and assert
#       np.isclose(P(|1>), np.sin((2*k+1)*theta2/2)**2).
# TODO: sample success counts at a few powers and re-run the MLAE grid scan.

In [ ]:
# Exercise 2 scaffold -- compare power schedules at fixed shots.
# for schedule in ([0], [0, 1], [0, 1, 2, 3, 4]):
#     hits_s = []
#     for k in schedule:
#         res = device.run(circuit_for_k(k), shots=shots).result()
#         hits_s.append(res.measurement_counts.get("1", 0))
#     TODO: redefine log_likelihood to loop over `schedule` and `hits_s`,
#           scan `grid`, and print a_hat and |a_hat - a_true| for each schedule.
#     TODO: comment on why more (and higher) powers reduce the error.

## Summary

- A single objective qubit prepared by $A = R_y(\theta)$ encodes the amplitude $a = \sin^2(\theta/2)$; here $a = 0.2$.
- The amplification operator $Q = A\,S_0\,A^\dagger\,S_f$ (with $S_f = Z$, $S_0 = XZX$, and $A^\dagger$ from `.adjoint()`) rotates the success probability to the exact value $p_k = \sin^2((2k+1)\theta/2)$ after $k$ steps -- verified from the state vector.
- Maximum-Likelihood Amplitude Estimation recovers $a$ from sampled success counts across several powers, with no controlled-$Q$ and no phase register -- ideal for qcsim's measure-all, big-endian semantics.
- Amplitude estimation's error scales as $O(1/N)$ in $A$-queries versus Monte Carlo's $O(1/\sqrt{N})$ -- a quadratic speedup confirmed by the log-log slopes $-1$ and $-1/2$.
- Reading $a$ as a discounted expected payoff turns the recovered amplitude directly into the fair price of a one-step binary asset.

**Next:** Continue to the [`04-quantum-ml`](../../04-quantum-ml/) track to apply variational circuits to machine-learning problems.